<a href="https://colab.research.google.com/github/Cyberpunk-San/ML-practice/blob/K-Fold-CV/K_Fold_CV_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

K-Fold CV Scratch

In [1]:
import numpy as np

def k_fold_cv_scratch(model_class, X, y, k=5, **kwargs):
    # 1. Shuffle the data indices to ensure randomness
    indices = np.arange(len(X))
    np.random.shuffle(indices)

    # 2. Split indices into k roughly equal parts
    folds = np.array_split(indices, k)
    fold_scores = []

    for i in range(k):
        # Identify validation and training indices for the current rotation
        val_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])

        # Split the actual data
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # 3. Initialize a fresh model instance and train it
        model = model_class(**kwargs)
        model.fit(X_train, y_train)

        # 4. Predict and calculate score (Accuracy for classification)
        preds = model.predict(X_val)
        accuracy = np.mean(preds == y_val)
        fold_scores.append(accuracy)

        print(f"Fold {i+1} Accuracy: {accuracy:.4f}")

    # Return the mean and standard deviation of all folds
    return np.mean(fold_scores), np.std(fold_scores)

Class K-Fold CV

In [2]:
from collections import Counter

class KNNScratch:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        # KNN is a "Lazy Learner" - it just stores the data
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        predictions = [self._predict(x) for x in X]
        return np.array(predictions)

    def _predict(self, x):
        # 1. Compute distances between x and all points in training set
        distances = [np.sqrt(np.sum((x - x_train)**2)) for x_train in self.X_train]

        # 2. Sort by distance and return indices of the first k neighbors
        k_indices = np.argsort(distances)[:self.k]

        # 3. Extract the labels of those k neighbors
        k_nearest_labels = [self.y_train[i] for i in k_indices]

        # 4. Return the most common label (Majority Vote)
        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]

To test on KNN

In [6]:
from sklearn.datasets import load_iris
iris = load_iris()
X, y = iris.data, iris.target

# Testing our CV logic on a hypothetical model
mean_acc, std_dev = k_fold_cv_scratch(KNNScratch, X, y, k=5)

print(f"\n--- Final CV Results ---")
print(f"Mean Accuracy: {mean_acc * 100:.2f}%")
print(f"Stability (Std Dev): {std_dev:.4f}")

Fold 1 Accuracy: 0.9333
Fold 2 Accuracy: 0.9667
Fold 3 Accuracy: 0.9667
Fold 4 Accuracy: 0.9333
Fold 5 Accuracy: 1.0000

--- Final CV Results ---
Mean Accuracy: 96.00%
Stability (Std Dev): 0.0249


Implementation

we generate a distribution of scores, giving us a Mean (expected performance) and a Standard Deviation (how much we can trust that performance). If the standard deviation is high, your model is "shaky" and prone to overfitting.